In [6]:
import wandb
import pandas as pd

# 1. Fetch the run
api = wandb.Api()
run = api.run("as7629-columbia-university/Multimodal-Nanochat/zgycvj5r")

# 2. Get history as a DataFrame
history_df = run.history()

# 3. Calculate the cumulative sum (first 21 dt entries)
warmup_time = history_df['train/dt'][1:21].sum()
print(warmup_time)
total_hot_time = history_df["total_training_time"].iloc[-1] - warmup_time
print(total_hot_time)
# 4. Save warmup_time and total_hot_time into the run summary
run.summary.update({'warmup_time': float(warmup_time), 'total_hot_time': float(total_hot_time)})



566.5089404582977
289.01555609703064


In [2]:
import wandb
import pandas as pd

# Initialize API
api = wandb.Api()

# Configuration
project_path = "as7629-columbia-university/Multimodal-Nanochat"
group_name = "data_pipeline_tuning_offline" # Replace with your group name

# 1. Fetch all runs in the project with the specified group
runs = api.runs(project_path, filters={"group": group_name})

print(f"Found {len(runs)} runs in group '{group_name}'...")

for run in runs:
    # 2. Get history (only fetch necessary keys to improve performance)
    history_df = run.history(keys=["train/mfu", "train/tok_per_sec", "_step"])

    # 3. Filter for steps 20-30 (inclusive)
    # Using _step ensures we target the actual training steps
    hot_zone = history_df[(history_df["_step"] >= 20) & (history_df["_step"] <= 30)]

    if not hot_zone.empty:
        # Calculate averages
        avg_mfu = hot_zone["train/mfu"].mean()
        avg_tok = hot_zone["train/tok_per_sec"].mean()

        # 4. Update the run summary
        run.summary["avg_hot_mfu"] = float(avg_mfu)
        run.summary["avg_hot_tok_per_sec"] = float(avg_tok)
        
        # Persist changes to the W&B server
        run.summary.update()
        print(f"Updated run {run.id}: avg_hot_mfu={avg_mfu:.4f}")
    else:
        print(f"Skipping run {run.id}: No data found for steps 20-30.")

Found 2 runs in group 'data_pipeline_tuning_offline'...
Updated run tzqkgfas: avg_hot_mfu=24.6549
Updated run hx9jqg2u: avg_hot_mfu=24.7993
